In [ ]:
# Ensure GPU runtime in Colab: Runtime → Change runtime type → GPU
!nvidia-smi || true

# Core libraries for parsing, RAG, and QLoRA fine-tuning
!pip -q install -U "transformers>=4.44" "accelerate>=0.33" "datasets>=2.20" \
  bitsandbytes peft trl sentencepiece protobuf==4.25.3 \
  pypdf rank-bm25 faiss-cpu orjson ujson \
  unstructured "unstructured[pdf]" tiktoken loralib sentence-transformers

import torch, platform
print("Torch CUDA:", torch.cuda.is_available(), "  GPUs:", torch.cuda.device_count())
print("Python:", platform.python_version())


Mon Jan  5 04:22:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# We'll use a project root inside Drive (feel free to change):
PROJECT_ROOT = "/content/drive/My Drive/Colab Notebooks/HuggingFace"
!mkdir -p "$PROJECT_ROOT/pdfs" "$PROJECT_ROOT/work"
PROJECT_ROOT


Mounted at /content/drive


'/content/drive/My Drive/Colab Notebooks/HuggingFace'

In [ ]:
!ls -l "$PROJECT_ROOT/TrainingData/"

total 92586
-rw------- 1 root root  3222628 Dec 10 02:54 '10 Relapse Prevention, Second Edition_ Maintenance Strategies in the Treatment of Addictive Behaviors-The Guilford Press (2005).pdf'
-rw------- 1 root root   501223 Dec 10 03:43  2004_Korn_Shaffer.pdf
-rw------- 1 root root 38056932 Dec 10 03:46  2025-Guidelines-For-Treating-Tobacco-Nicotine-Dependance-English-v2.pdf
-rw------- 1 root root  1288459 Dec 10 03:45  clinical-guidance-withdrawal-alcohol-and-other-drugs.pdf
-rw------- 1 root root  1700935 Dec 10 03:47  Guide_E.pdf
-rw------- 1 root root   324782 Dec 10 03:44  Guidelines_for_the_Prevention_and_Treatment_of_Ben.pdf
-rw------- 1 root root  3040625 Dec 10 03:44  interventions-approaches-and-guidelines-for-gambling-an-evidence-review-2025.pdf
-rw------- 1 root root   728859 Dec 10 03:47  match03.pdf
-rw------- 1 root root 12016983 Dec 10 02:57  material_02a.pdf
-rw------- 1 root root   221670 Dec 10 03:48  Principles_of_Drug_Addiction_Treatment_A_Research-Based_Guide_Third

In [ ]:
# Install poppler-utils for PDF processing
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
#Parse PDFs

!python "$PROJECT_ROOT/therapist_pipeline.py" parse-pdfs \
  --src "$PROJECT_ROOT/TrainingData" \
  --out "$PROJECT_ROOT/work/corpus_raw.jsonl"

!wc -l "$PROJECT_ROOT/work/corpus_raw.jsonl" || true
!sed -n '1,2p' "$PROJECT_ROOT/work/corpus_raw.jsonl"

2026-01-04 09:55:36.729395: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767520536.746550   18964 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767520536.751403   18964 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767520536.764225   18964 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767520536.764245   18964 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767520536.764248   18964 computation_placer.cc:177] computation placer alr

The `unstructured` library, used for PDF parsing, requires the `poppler-utils` package. This package provides tools like `pdfinfo` that are necessary to extract information from PDF documents. Let's install it.

Let's confirm the content of the `therapist_pipeline.py` file that has been saved to your Google Drive:

In [ ]:
#!cat "$PROJECT_ROOT/therapist_pipeline.py"

# sud_therapist_pipeline.py
# -*- coding: utf-8 -*-
'''
End-to-end pipeline to build a privacy-preserving, empathetic SUD counselor:
- Parse open-access PDFs -> raw JSONL
- Build chunks + instruction-tuning pairs (MI/CBT/DBT)
- Build FAISS retrieval index (RAG)
- Fine-tune (QLoRA) a Hugging Face model
- Inference loop with RAG grounding + hard-coded crisis hotlines (US/UK/EU/Global)

Intended paths for the accompanying Colab notebook:
- Project root: /HuggingFace
- PDFs: /TrainingData
- Artifacts: /HuggingFace/work
'''

from __future__ import annotations
import os, re, json, glob, argparse, random
from pathlib import Path
from typing import Optional, Dict, List

# -----------------------------
# SAFETY & CRISIS ESCALATION
# -----------------------------
CRISIS_PATTERNS: List[re.Pattern] = [
    re.compile(r"\b(kill myself|end my life|suicide|suicidal|take my own life|self[-\s]?harm|cutting|i don't want to live|can('?|no)t go on)\b", re.I),
    re.compile(r"\b(overdose|OD|o\.d\.|took to

In [ ]:
content = """# sud_therapist_pipeline.py
# -*- coding: utf-8 -*-
'''
End-to-end pipeline to build a privacy-preserving, empathetic SUD counselor:
- Parse open-access PDFs -> raw JSONL
- Build chunks + instruction-tuning pairs (MI/CBT/DBT)
- Build FAISS retrieval index (RAG)
- Fine-tune (QLoRA) a Hugging Face model
- Inference loop with RAG grounding + hard-coded crisis hotlines (US/UK/EU/Global)

Intended paths for the accompanying Colab notebook:
- Project root: /HuggingFace
- PDFs: /TrainingData
- Artifacts: /HuggingFace/work
'''

from __future__ import annotations
import os, re, json, glob, argparse, random
from pathlib import Path
from typing import Optional, Dict, List

# -----------------------------
# SAFETY & CRISIS ESCALATION
# -----------------------------
CRISIS_PATTERNS: List[re.Pattern] = [
    re.compile(r"\\b(kill myself|end my life|suicide|suicidal|take my own life|self[-\\s]?harm|cutting|i don't want to live|can('?|no)t go on)\\b", re.I),
    re.compile(r"\\b(overdose|OD|o\\.d\\.|took too many|too much [a-z]+|not breathing|blue lips|passed out)\\b", re.I),
    re.compile(r"\\b(right now|tonight|this (?:morning|afternoon|evening)|today)\\b.*\\b(kill|overdose|end it|harm myself)\\b", re.I),
    re.compile(r"\\b(gun|firearm|pills|fentanyl|heroin|poison|rope|knife)\\b.*\\b(now|ready|tonight|today)\\b", re.I),
    re.compile(r"\\b(seizure|delirium tremens|DTs|shaking uncontrollably|hallucinating|confused and hallucinating)\\b", re.I),
]

def needs_crisis_escalation(text: str) -> bool:
    if not text or not text.strip():
        return False
    return any(p.search(text) for p in CRISIS_PATTERNS)

HOTLINES: Dict[str, Dict[str, str]] = {
    "US": {
        "emergency": "If you are in immediate danger, call **911**.",
        "lifeline": "988 Suicide & Crisis Lifeline \u2014 call or text **988** (24/7, free & confidential).",
        "textline": "Crisis Text Line \u2014 text **HOME** to **741741** (24/7).",
        "more": "More info: 988lifeline.org \u2022 crisistextline.org"
    },
    "UK": {
        "emergency": "If you are in immediate danger, call **999** (or **112**).",
        "samaritans": "Samaritans \u2014 call **116 123** (24/7, free).",
        "shout": "Shout \u2014 text **SHOUT** to **85258** (24/7).",
        "papyrus": "HOPELINEUK (young people) \u2014 **0800 068 4141**.",
        "more": "See: samaritans.org \u2022 NHS urgent help pages"
    },
    "EU": {
        "emergency": "If you are in immediate danger, call **112** (EU-wide emergency).",
        "directory": "Befrienders Worldwide \u2014 visit **https://www.befrienders.org** to find your local hotline.",
        "telefon": "In Germany/Austria: Telefonseelsorge \u2014 **0800-1110111** / **0800-1110222**; in many EU countries: **116 123**.",
        "more": "Check your national health ministry pages for local 24/7 lines."
    },
    "GLOBAL": {
        "emergency": "If you are in immediate danger, call your local emergency number (e.g., **112/999/911**).",
        "directory": "Befrienders Worldwide \u2014 **https://www.befrienders.org** to find a local hotline.",
        "more": "If internet/phone access is limited, seek in-person help from someone nearby."
    }
}

def detect_region(user_profile: Optional[Dict] = None, text: Optional[str] = None) -> str:
    if isinstance(user_profile, dict):
        r = (user_profile.get("region") or user_profile.get("country") or "").upper()
        if "UNITED STATES" in r or r in {"US", "USA"} or "AMERICA" in r:
            return "US"
        if r in {"UK", "UNITED KINGDOM", "ENGLAND", "SCOTLAND", "WALES", "NORTHERN IRELAND"}:
            return "UK"
        EU_SET = {
            "EU","EUROPE","GERMANY","FRANCE","SPAIN","ITALY","NETHERLANDS","BELGIUM","SWEDEN","NORWAY",
            "DENMARK","FINLAND","AUSTRIA","POLAND","PORTUGAL","GREECE","IRELAND","CZECH","CZECHIA","HUNGARY",
            "SLOVAKIA","SLOVENIA","LITHUANIA","LATVIA","ESTONIA","CROATIA","ROMANIA","BULGARIA","LUXEMBOURG",
            "MALTA","CYPRUS"
        }
        if r in EU_SET:
            return "EU"
    hay = (text or "").lower()
    if any(k in hay for k in ["united states","usa","u.s.","u.s.a","america"]) or "911" in hay or "988" in hay:
        return "US"
    if any(k in hay for k in ["united kingdom","england","scotland","wales","northern ireland"]) or \\
       any(k in hay for k in ["999","116 123","85258"]):
        return "UK"
    if "112" in hay or any(k in hay for k in [
        "europe","germany","france","spain","italy","netherlands","belgium","sweden","norway","denmark",
        "finland","austria","poland","portugal","greece","ireland","czech","czechia","hungary","slovakia",
        "slovenia","lithuania","latvia","estonia","croatia","romania","bulgaria","luxembourg","malta","cyprus"
    ]):
        return "EU"
    return "GLOBAL"

def hotline_payload(region: str) -> Dict[str, str]:
    region = (region or "").upper()
    return HOTLINES.get(region, HOTLINES["GLOBAL"])

def format_hotline_message(region: str) -> str:
    data = hotline_payload(region)
    order = ["emergency","lifeline","textline","samaritans","shout","papyrus","directory","telefon","more"]
    lines = [
        "I\u2019m detecting signs of a possible crisis. Your safety comes first.",
        "Please consider contacting **real-time crisis support** now:",
        ""
    ]
    for k in order:
        if k in data:
            lines.append(f"- {data[k]}")
    lines += ["", "If you can, stay with someone you trust. You can return here anytime after you\u2019re safe."]
    return "\\n".join(lines)

def safety_gate(user_profile: Optional[Dict], user_text: str) -> Optional[str]:
    if not needs_crisis_escalation(user_text or ""):
        return None
    region = detect_region(user_profile, user_text)
    return format_hotline_message(region)

# -----------------------------
# PDF -> RAW JSONL
# -----------------------------
def cmd_parse_pdfs(args):
    from unstructured.partition.pdf import partition_pdf

    def clean(txt: str) -> str:
        txt = re.sub(r'\\s+', ' ', txt).strip()
        txt = re.sub(r'(Page\\s+\\d+)|(^\\d+\\s*$)', '', txt)
        return txt

    def pdf_to_text(pdf_path: str) -> str:
        elements = partition_pdf(pdf_path, infer_table_structure=True, strategy="hi_res")
        return clean(" ".join([e.text for e in elements if hasattr(e, "text")]))

    src = Path(args.src)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as w:
        for pdf in glob.glob(str(src / "*.pdf")):
            text = pdf_to_text(pdf)
            rec = {"source": os.path.basename(pdf), "text": text}
            w.write(json.dumps(rec, ensure_ascii=False) + "\\n")
    print(f"Wrote: {out}")

# -----------------------------
# RAW -> CHUNKS + SFT PAIRS
# -----------------------------
SUBSTANCE_TAGS = {
    "alcohol":["alcohol","AUD"],
    "opioids":["opioid","heroin","fentanyl","OUD"],
    "stimulants":["cocaine","methamphetamine","stimulant","crystal meth","amphetamine"],
    "cannabis":["cannabis","marijuana","THC"],
    "gambling":["gambling","betting","wager"],
    "benzos":["benzodiazepine","alprazolam","clonazepam","lorazepam","diazepam"],
    "smoking":["nicotine","smoking","tobacco","cigarette","vaping"],
}

MI_PROMPTS = [
    "Use motivational interviewing to explore ambivalence and elicit change talk.",
    "Reflect, affirm, and summarize; end with a collaborative next step.",
]
CBT_PROMPTS = [
    "Identify triggers, thoughts, feelings, behaviors; propose coping skills.",
    "Propose a relapse-prevention plan with high-risk situations and coping responses."
]
DBT_PROMPTS = [
    "Apply distress tolerance and emotion regulation skills.",
    "Offer mindful, nonjudgmental steps and interpersonal effectiveness suggestions."
]

def split_into_chunks(text: str, max_chars: int = 1200, min_chars: int = 400) -> List[str]:
    chunks, i = [], 0
    L = len(text)
    while i < L:
        j = min(i + max_chars, L)
        k = text.rfind(". ", i, j)
        if k == -1 or k < i + min_chars:
            k = j
        chunk = text[i:k].strip()
        if len(chunk) >= min_chars:
            chunks.append(chunk)
        i = k + 1
    return chunks

def detect_tags(txt: str) -> List[str]:
    tags = set()
    low = txt.lower()
    for tag, kws in SUBSTANCE_TAGS.items():
        if any(kw.lower() in low for kw in kws):
            tags.add(tag)
    return list(tags) or ["general"]

def synthesize_instruction(chunk: str, style: str) -> Dict[str, str]:
    if style == "MI":
        sys = "You are an empathetic counselor using Motivational Interviewing (MI)."
        inst = random.choice(MI_PROMPTS)
    elif style == "CBT":
        sys = "You are an empathetic counselor using Cognitive Behavioral Therapy (CBT) and relapse prevention."
        inst = random.choice(CBT_PROMPTS)
    else:
        sys = "You are an empathetic counselor using Dialectical Behavior Therapy (DBT)."
        inst = random.choice(DBT_PROMPTS)

    # Construct the triple quotes dynamically to avoid syntax conflicts with the outer 'content' string
    triple_double_quotes_str = chr(34) + chr(34) + chr(34)
    user = f"I\u2019m struggling with urges. Based on the following guidance, help me plan next steps:\\n\\n{triple_double_quotes_str}{chunk[:900]}{triple_double_quotes_str}"
    assistant = (
        "Let's work through this together. First, I\u2019ll reflect what I\u2019m hearing; "
        "then we\u2019ll outline practical steps, and end with a small, achievable commitment for today."
    )
    return {"system": sys, "instruction": inst, "input": user, "output": assistant}

def cmd_build_examples(args):
    raw = Path(args.raw)
    chunks_path = Path(args.chunks); chunks_path.parent.mkdir(parents=True, exist_ok=True)
    sft_path = Path(args.sft); sft_path.parent.mkdir(parents=True, exist_ok=True)

    with raw.open("r", encoding="utf-8") as f, \
         chunks_path.open("w", encoding="utf-8") as c_out, \
         sft_path.open("w", encoding="utf-8") as sft_out:
        for line in f:
            doc = json.loads(line)
            for ch in split_into_chunks(doc["text"]):
                tags = detect_tags(ch)
                c_out.write(json.dumps({"source": doc["source"], "text": ch, "tags": tags}, ensure_ascii=False) + "\\n")
                for style in ["MI", "CBT", "DBT"]:
                    ex = synthesize_instruction(ch, style)
                    ex["tags"] = tags
                    sft_out.write(json.dumps(ex, ensure_ascii=False) + "\\n")
    print(f"Wrote: {chunks_path} and {sft_path}")

# -----------------------------
# BUILD FAISS INDEX (RAG)
# -----------------------------
def cmd_build_faiss(args):
    import faiss
    from sentence_transformers import SentenceTransformer

    chunks = Path(args.chunks)
    index_dir = Path(args.index); index_dir.mkdir(parents=True, exist_ok=True)

    texts, metas = [], []
    with chunks.open("r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            texts.append(j["text"])
            metas.append(j)

    model = SentenceTransformer(args.embed)
    emb = model.encode(texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, str(index_dir / "index.faiss"))

    with (index_dir / "metas.jsonl").open("w", encoding="utf-8") as w:
        for m in metas:
            w.write(json.dumps(m, ensure_ascii=False) + "\\n")

    print(f"FAISS index written to {index_dir}")

# -----------------------------
# TRAIN (QLoRA SFT)
# -----------------------------
def cmd_train_sft(args):
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer

    out_dir = Path(args.out); out_dir.mkdir(parents=True, exist_ok=True)
    ds = load_dataset("json", data_files=args.sft, split="train").train_test_split(test_size=0.02, seed=42)

    tok = AutoTokenizer.from_pretrained(args.base, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="bfloat16"
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.base,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True
    )

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    )
    model = get_peft_model(model, lora_cfg)

    def format_example(ex):
        return {
            "text": (
                f"<s>[SYSTEM]\\n{ex['system']}\\n\\n"
                f"[INSTRUCTION]\\n{ex['instruction']}\\n\\n"
                f"[USER]\\n{ex['input']}\\n\\n"
                f"[ASSISTANT]\\n{ex['output']}"
            )
        }

    train_ds = ds["train"].map(lambda b: {"text": [format_example({col: b[col][i] for col in b})["text"] for i in range(len(b[list(b.keys())[0]])) ]},
                               batched=True, remove_columns=ds["train"].column_names)
    eval_ds  = ds["test"].map(lambda b: {"text": [format_example({col: b[col][i] for col in b})["text"] for i in range(len(b[list(b.keys())[0]])) ]},
                              batched=True, remove_columns=ds["test"].column_names)

    tr_args = TrainingArguments(
        output_dir=str(out_dir),
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        logging_steps=20,
        eval_steps=200,
        save_steps=500,
        bf16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        warmup_steps=100,
        save_total_limit=3,
        report_to="none"
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=tr_args,
        # Removed dataset_text_field as it's deprecated
        # Removed tokenizer as it's deprecated
        # Removed max_seq_length as it's deprecated
        # Removed packing as it's deprecated
    )
    trainer.train()

    trainer.model.save_pretrained(str(out_dir))
    tok.save_pretrained(str(out_dir))
    print(f"SFT adapter saved to {out_dir}")

# -----------------------------
# INFERENCE LOOP (RAG + SAFETY)
# -----------------------------
def cmd_infer(args):
    import faiss
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
    from peft import PeftModel

    index_dir = Path(args.index)
    index = faiss.read_index(str(index_dir / "index.faiss"))
    metas = [json.loads(x) for x in (index_dir / "metas.jsonl").read_text(encoding="utf-8").splitlines()]
    emb_model = SentenceTransformer(args.embed)

    def retrieve(q_text, k=4):
        q = emb_model.encode([q_text], normalize_embeddings=True)
        _, I = index.search(q, k)
        return [metas[i]["text"] for i in I[0]]

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="bfloat16", bnb_4bit_quant_type="nf4")
    tok = AutoTokenizer.from_pretrained(args.base, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        args.base,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, args.adapter)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=False
    )

    SAFETY_POLICY = (
        "You are a supportive, nonjudgmental counselor. You are not a doctor.\\n"
        "You never give diagnoses or prescribe medication. Use evidence-based skills (MI/CBT/DBT).\\n"
        "Encourage seeking professional help. If you detect crisis (self-harm, overdose), provide crisis resources.\\n"
    )

    def make_prompt(user_msg: str, substance_hint: str = "general") -> str:
        passages = retrieve(user_msg, k=4)
        context = "\\n\\n".join(f"- {p[:700]}" for p in passages)
        sys = (
            f"[SYSTEM]\\n{SAFETY_POLICY}\\n"
            f"Substance focus: {substance_hint}.\\n"
            f"Ground your suggestions in the following vetted materials:\\n{context}\\n"
        )
        return f"<s>{sys}\\n[USER]\\n{user_msg}\\n\\n[ASSISTANT]\\n"

    print("Therapeutic SUD Counselor \u2014 type 'exit' to quit.")
    user_profile = {"region": args.region} if args.region else None

    while True:
        user = input("User: ").strip()
        if user.lower() in {"exit", "quit"}:
            break

        crisis_msg = safety_gate(user_profile, user)
        if crisis_msg:
            print("\\nAssistant:\\n" + crisis_msg + "\\n")
            continue

        prompt = make_prompt(user, args.substance)
        out = pipe(prompt)[0]["generated_text"].split("[ASSISTANT]")[-1].strip()
        print("\\nAssistant:", out, "\\n")

# -----------------------------
# ARGPARSE
# -----------------------------
def main():
    p = argparse.ArgumentParser(description="SUD Counselor Pipeline")
    sub = p.add_subparsers(dest="cmd", required=True)

    sp = sub.add_parser("parse-pdfs", help="Extract text from PDFs to raw JSONL")
    sp.add_argument("--src", required=True, help="Directory with PDFs")
    sp.add_argument("--out", required=True, help="Output JSONL")
    sp.set_defaults(func=cmd_parse_pdfs)

    sb = sub.add_parser("build-examples", help="Build chunks + SFT pairs")
    sb.add_argument("--raw", required=True, help="Input raw JSONL")
    sb.add_argument("--chunks", required=True, help="Output chunks JSONL")
    sb.add_argument("--sft", required=True, help="Output SFT JSONL")
    sb.set_defaults(func=cmd_build_examples)

    sf = sub.add_parser("build-faiss", help="Create FAISS index for RAG")
    sf.add_argument("--chunks", required=True, help="Chunks JSONL")
    sf.add_argument("--index", required=True, help="Output index dir")
    sf.add_argument("--embed", default="sentence-transformers/all-MiniLM-L6-v2", help="Embedding model")
    sf.set_defaults(func=cmd_build_faiss)

    st = sub.add_parser("train-sft", help="Train QLoRA SFT adapter")
    st.add_argument("--base", required=True, help="Base HF model")
    st.add_argument("--sft", required=True, help="SFT pairs JSONL")
    st.add_argument("--out", required=True, help="Output adapter dir")
    st.set_defaults(func=cmd_train_sft)

    si = sub.add_parser("infer", help="RAG-grounded inference loop with safety gate")
    si.add_argument("--base", required=True, help="Base HF model")
    si.add_argument("--adapter", required=True, help="Trained adapter dir")
    si.add_argument("--index", required=True, help="FAISS index dir")
    si.add_argument("--embed", default="sentence-transformers/all-MiniLM-L6-v2", help="Embedding model")
    si.add_argument("--substance", default="general", help="Substance hint for system prompt")
    si.add_argument("--region", default=None, help="Default region hint: US/UK/EU")
    si.set_defaults(func=cmd_infer)

    args = p.parse_args()
    args.func(args)

if __name__ == "__main__":
    main()"""

with open(f"{PROJECT_ROOT}/therapist_pipeline.py", "w", encoding="utf-8") as f:
    f.write(content)

print(f"Content written to {PROJECT_ROOT}/therapist_pipeline.py")

Content written to /content/drive/My Drive/Colab Notebooks/HuggingFace/therapist_pipeline.py


In [ ]:
"""
#Parse PDFs

!python "$PROJECT_ROOT/therapist_pipeline.py" parse-pdfs \
  --src /HuggingFace/TrainingData \
  --out "$PROJECT_ROOT/work/corpus_raw.jsonl"

!wc -l "$PROJECT_ROOT/work/corpus_raw.jsonl" || true
!sed -n '1,2p' "$PROJECT_ROOT/work/corpus_raw.jsonl"

Wrote: /content/drive/My Drive/Colab Notebooks/HuggingFace/work/corpus_raw.jsonl
0 /content/drive/My Drive/Colab Notebooks/HuggingFace/work/corpus_raw.jsonl


In [ ]:
# Display the content of therapist_pipeline.py to inspect its code
#!cat "$PROJECT_ROOT/therapist_pipeline.py"

In [ ]:
#Build chunks and SFT pairs
!python "$PROJECT_ROOT/therapist_pipeline.py" build-examples \
  --raw "$PROJECT_ROOT/work/corpus_raw.jsonl" \
  --chunks "$PROJECT_ROOT/work/corpus_chunks.jsonl" \
  --sft "$PROJECT_ROOT/work/sft_pairs.jsonl"

!wc -l "$PROJECT_ROOT/work/corpus_chunks.jsonl" "$PROJECT_ROOT/work/sft_pairs.jsonl" || true

Wrote: /content/drive/My Drive/Colab Notebooks/HuggingFace/work/corpus_chunks.jsonl and /content/drive/My Drive/Colab Notebooks/HuggingFace/work/sft_pairs.jsonl
    4097 /content/drive/My Drive/Colab Notebooks/HuggingFace/work/corpus_chunks.jsonl
   12291 /content/drive/My Drive/Colab Notebooks/HuggingFace/work/sft_pairs.jsonl
   16388 total


In [ ]:
#build FAISS (RAG)
!python "$PROJECT_ROOT/therapist_pipeline.py" build-faiss \
  --chunks "$PROJECT_ROOT/work/corpus_chunks.jsonl" \
  --index "$PROJECT_ROOT/work/faiss_index" \
  --embed sentence-transformers/all-MiniLM-L6-v2

!ls -al "$PROJECT_ROOT/work/faiss_index"

2026-01-05 04:24:05.192584: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 04:24:05.209845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767587045.230903    8166 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767587045.237350    8166 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767587045.253630    8166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from huggingface_hub import login
# Generate a READ token at https://huggingface.co/settings/tokens
login()


In [ ]:
#Train QLORA Adaptor

BASE = "ArkMaster123/qwen2.5-7b-therapist"  # or another base from your shortlist

!python "$PROJECT_ROOT/therapist_pipeline.py" train-sft \
  --base "$BASE" \
  --sft  "$PROJECT_ROOT/work/sft_pairs.jsonl" \
  --out  "$PROJECT_ROOT/work/ArkMaster-sud-sft"

2026-01-05 04:24:51.791762: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 04:24:51.808679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767587091.829611    8471 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767587091.836009    8471 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767587091.851674    8471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Here are some popular base models that could be used for fine-tuning. You can replace the value of `BASE` in the previous cell with one of these, or another model ID you verify on [Hugging Face Models](https://huggingface.co/models).

- `meta-llama/Llama-2-7b-chat-hf` (requires access)
- `mistralai/Mistral-7B-Instruct-v0.2`
- `HuggingFaceH4/zephyr-7b-beta`
- `microsoft/Phi-3-mini-4k-instruct` (smaller, but efficient)

In [ ]:
#infer and test the model with some prompts
!python "$PROJECT_ROOT/therapist_pipeline.py" infer \
  --base    "$BASE" \
  --adapter "$PROJECT_ROOT/work/ArkMaster-sud-sft" \
  --index   "$PROJECT_ROOT/work/faiss_index" \
  --embed   sentence-transformers/all-MiniLM-L6-v2 \
  --substance general \
  --region  US

2026-01-05 07:11:04.548804: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 07:11:04.566900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767597064.588582   49696 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767597064.595208   49696 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767597064.611737   49696 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
#Save model and parsed pdfs

import shutil, os, pathlib

DRIVE_ROOT = "/content/drive/My Drive/Colab Notebooks/HuggingFace/sud_counselor"
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

# Adapter (trained output)
shutil.copytree(
    "/content/drive/My Drive/Colab Notebooks/HuggingFace/work/ArkMaster-sud-sft",
    f"{DRIVE_ROOT}/ArkMaster-sud-sft",
    dirs_exist_ok=True
)

# (Optional) Save your FAISS RAG index too
shutil.copytree(
    "/content/drive/My Drive/Colab Notebooks/HuggingFace/work/faiss_index",
    f"{DRIVE_ROOT}/faiss_index",
    dirs_exist_ok=True
)

# (Optional) Save corpora (so you don't re-parse PDFs)
for fn in ["corpus_raw.jsonl", "corpus_chunks.jsonl", "sft_pairs.jsonl"]:
    src = f"/content/drive/My Drive/Colab Notebooks/HuggingFace/work/{fn}"
    if os.path.exists(src):
        shutil.copy2(src, f"{DRIVE_ROOT}/{fn}")

print("Saved to:", DRIVE_ROOT)
!ls -al "$DRIVE_ROOT"


Saved to: /content/drive/My Drive/Colab Notebooks/HuggingFace/sud_counselor
total 25991
drwx------ 5 root root     4096 Jan  5 06:17 ArkMaster-sud-sft
-rw------- 1 root root  4936885 Jan  5 04:23 corpus_chunks.jsonl
-rw------- 1 root root  4485757 Jan  4 12:22 corpus_raw.jsonl
drwx------ 2 root root     4096 Jan  5 04:24 faiss_index
-rw------- 1 root root 17182542 Jan  5 04:23 sft_pairs.jsonl


In [ ]:
#Re load model (in new colab session)

import shutil, pathlib
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/My Drive/Colab Notebooks/HuggingFace/sud_counselor"
pathlib.Path("/HuggingFace/work").mkdir(parents=True, exist_ok=True)

shutil.copytree(f"{DRIVE_ROOT}/ArkMaster-sud-sft", "/HuggingFace/work/ArkMaster-sud-sft", dirs_exist_ok=True)
shutil.copytree(f"{DRIVE_ROOT}/faiss_index", "/HuggingFace/work/faiss_index", dirs_exist_ok=True)

!ls -al /HuggingFace/work/haizea-sud-sft
